# Analysis of the IDHKP-IDH network using ToricVerticalSystems.jl

In this notebook, we illustrate the functionality of our package `ToricVerticalSystems.jl` for the IDHKP-IDH network (Example 1.1 and Figure 7.1(a) in the paper):

<img src="images/idh_network.png" alt="drawing" width="500"/>

This network modeles the regulation of the carbon flow between the Krebs cycle glyoxylate bypass in bacterial cells, and has previously appeared in, e.g., work of Shinar and Feinberg on absolute concentration robustness.

In [1]:
using ToricVerticalSystems, Oscar, Graphs;

We begin by giving the defining matrices for the network:

In [2]:
# Stoichiometric matrix
N = matrix(QQ, [[-1, 1, 1, 0, 0, 0],
     [-1, 1, 0, 0, 0, 1],
     [1, -1, -1, -1, 1, 1],
     [0, 0, 1, -1, 1, 0],
     [0, 0, 0, 1, -1, -1]])

# Reactant matrix
M = matrix(ZZ, [[1, 0, 0, 0, 0, 0],
     [1, 0, 0, 0, 0, 0],
     [0, 1, 1, 1, 0, 0],
     [0, 0, 0, 1, 0, 0],
     [0, 0, 0, 0, 1, 1]])

[1   0   0   0   0   0]
[1   0   0   0   0   0]
[0   1   1   1   0   0]
[0   0   0   1   0   0]
[0   0   0   0   1   1]

In [3]:
n = nrows(M) # Number of species
r = ncols(M) # Number of reactions
s = rank(N) # Rank
d = n-s # Corank
println("n=$n, r=$r, d=$d")

n=5, r=6, d=2


In [4]:
# Row reduced stoichiometric matrix
C = row_space(N)

# Conserved quantities
W = kernel(N, side=:left)

[-2   1   -1   1   0]
[ 1   0    1   0   1]

## The steady state system

In [5]:
# Consistency
nonempty_positive_kernel(N)

true

In [6]:
# Generic nondegeneracy
has_nondegenerate_zero(N, M)

true

## Graph structure

In [7]:
# Graph structure and deficiency
g, embedding = reaction_graph(N, M)

# Complexes
m = complexes(N, M) |> length
println("m=$m")

# Linkage classes
ell = linkage_classes(N, M) |> length
println("ell=$ell")



m=6
ell=2


## Deficiency theory

In [8]:
# Deficiency
deficiency  = delta(N, M)
println("deficiency=$deficiency")

deficiency=1


In [9]:
covered_by_deficiency_zero_theorem(N, M)

false

In [10]:
covered_by_deficiency_one_theorem(N, M)

false

## Intermediates

In [11]:
# Intermediates
intermediates_result = intermediates(N, M, only_single_input=true)

1-element Vector{@NamedTuple{species::Int64, input_complexes::Vector{Vector{Int64}}, output_complexes::Vector{Vector{Int64}}}}:
 (species = 5, input_complexes = [[0, 0, 1, 1, 0]], output_complexes = [[0, 1, 1, 0, 0], [0, 0, 1, 1, 0]])

In [12]:
# Reduced network
Ntilde, Mtilde = reduced_network(N, M, intermediates_result)
Ctilde = row_space(Ntilde)

[1   -1   0   -1]
[0    0   1   -1]

## Toric invariance group

In [13]:
# Toric invariance group
Atilde = toric_invariance_group(Ctilde, Mtilde)

[1   0   1   0]
[0   1   1   0]

In [14]:
# Lifted toric invariance group
A = lift_exponent_matrix(Atilde, M, intermediates_result)

[1   0   1   0   1]
[0   1   1   0   1]

In [15]:
# Sanity check
A_original = toric_invariance_group(C, M)
row_space(A) == row_space(A_original)

true

In [16]:
# Does the toric invariance group consist of conserved quantities?
all(is_zero, A*N)

false

## Coset counting

In [17]:
# Injectivity test
injectivity_test(Ctilde, Mtilde, Atilde)

true

In [18]:
# Finitely many cosets for all rate constants?
all_positive_roots_nondegenerate(Ctilde, Mtilde, QQ.(Atilde))

true

In [19]:
# Mixed volume bound on the number of cosets
F = coset_counting_system(Ctilde, Mtilde, Atilde)
mixed_volume(F)

2

In [20]:
# Constant number of cosets for all rate constants?
N_A_tilde = reconstruct_stoichiometric_matrix_with_row_space_and_conserved_quantities(Ctilde,QQ.(Atilde))
display(minimal_siphons(N_A_tilde, Mtilde))
siphon_test(N_A_tilde, Mtilde)

3-element Vector{Set{Int64}}:
 Set([2, 3, 1])
 Set([3, 1])
 Set([2, 3])

true

In [21]:
positive_vector_in_rowspace(Atilde)

false

In [22]:
# Number of positive solutions for random parameters
number_of_cosets_for_random_parameters(Ctilde, Mtilde, Atilde; certify=true)

1

In [23]:
# Putting it all together
coset_counting_analysis(Ntilde, Mtilde, Atilde)

Injectivity test passed for reduced network
Toricity!


(toricity = true, finite = true)

## Check for binomiality

In [24]:
# Gröbner basis test for generic binomiality
F = vertical_system(C, M)
binomiality_check(F, verbose=true)

Gröbner basis: Generically binomial!
Specialization for all rate constants: Verified


(generically = true, for_all_positive = true)

## Applications of toric structure

In [25]:
# Capacity for multistationarity for original network
coset_with_multistationarity(A, W)

false

In [26]:
# (Local) ACR check
zero_columns(A)

1-element Vector{Int64}:
 4

In [27]:
# Invariants
A_circuits = matroid_from_matrix_columns(A) |> circuits 

4-element Vector{Vector{Int64}}:
 [4]
 [3, 5]
 [1, 2, 3]
 [1, 2, 5]

In [28]:
# Invariants for {1,2,3}
kernel(A[:, [1, 2, 3]], side=:right)

[-1]
[-1]
[ 1]